ROSP-3: Alert Generation Timing
======

In [ ]:
import numpy as np
import datetime
from lsst_efd_client import EfdClient
import astropy.time
from collections import Counter, defaultdict
import lsst.daf.butler as dafButler

In [ ]:
# day_obs_start = "20261102"
# day_obs_end = "20261109"

In [ ]:
print(f"{day_obs_start=}")
print(f"{day_obs_end=}")

In [ ]:
def dayobs_to_datetime(dayobs_string: str, start=True):
    """
    Return an astropy Time corresponding to the beginning of the day_obs if
    start is True (default), or corresponding to the end of the day_obs if start
    is False.
    """

    delta = (
        datetime.timedelta()
        if start
        else datetime.timedelta(hours=24) - datetime.timedelta(seconds=1)
    )
    return astropy.time.Time(
        datetime.datetime(
            int(dayobs_string[:4]),
            int(dayobs_string[4:6]),
            int(dayobs_string[6:8]),
            12,
            0,
            0,
        )
        + delta
    )


def datetime_to_dayobs(t: astropy.time.Time):
    return t.strftime("%Y%m%d")


def dayobs_range(start: str, end: str):
    """
    Return a list of day_obs values between start and end, inclusive start and
    end days.
    """
    start_datetime = dayobs_to_datetime(start)
    end_datetime = dayobs_to_datetime(end)
    n_days = int(
        np.ceil(
            (end_datetime - start_datetime)
            / astropy.time.TimeDelta(datetime.timedelta(hours=24))
        )
    )
    return [
        datetime_to_dayobs(start_datetime + datetime.timedelta(hours=24 * n))
        for n in range(n_days + 1)
    ]


async def collect_timing_stats_for_range(
    start: astropy.time.Time, end: astropy.time.Time, efd_client
):
    output = {}

    pp_query = efd_client.build_time_range_query(
        "lsst.prompt.prod.associationTimingMetrics",
        ["alert_timing_since_shutter_close"],
        start,
        end,
    )
    pp_res = await efd_client.influxql_query(pp_query)

    output["alert_visit_count"] = len(pp_res)
    output["median_last_alert_time"] = np.median(
        pp_res["alert_timing_since_shutter_close"]
    )
    output["alert_frac_under_120s"] = np.sum(
        pp_res["alert_timing_since_shutter_close"] < 120
    ) / len(pp_res)
    output["alert_frac_under_180s"] = np.sum(
        pp_res["alert_timing_since_shutter_close"] < 180
    ) / len(pp_res)

    return output


async def collect_visit_stats_for_dayobs(day_obs, efd_client, butler):
    #
    # Leaving this as a per-dayobs function because that's the simplest way to
    # make sure the EFD data and the Butler data are looking at matching sets of visits.
    #
    output = {}

    start = dayobs_to_datetime(day_obs)
    end = dayobs_to_datetime(day_obs, start=False)
    query = efd_client.build_time_range_query(
        "lsst.prompt.prod.initialPviSummaryMetrics", ["visit"], start, end
    )
    visit_metrics = await efd_client.influxql_query(query)
    if len(visit_metrics) == 0:
        visit_count = Counter()
    else:
        visit_count = Counter(visit_metrics["visit"])

    exposure_records = butler.query_dimension_records(
        "exposure",
        instrument="LSSTCam",
        where=f"day_obs={day_obs} AND (exposure.can_see_sky or exposure.can_see_sky=NULL) AND exposure.observation_type='science'",
        explain=False,
        limit=None,
    )

    output["prompt_visit_count"] = len(visit_count)
    output["prompt_detector_count"] = len(visit_metrics)
    output["butler_exposure_count"] = len(exposure_records)

    return output

In [ ]:
client = EfdClient("usdf_efd", db_name="lsst.prompt")
b = dafButler.Butler("embargo")

Overall Alert Generation Time
-----

In [ ]:
start = dayobs_to_datetime(f"{day_obs_start}")
end = dayobs_to_datetime(f"{day_obs_end}", start=False)

timing_stats = await collect_timing_stats_for_range(start, end, client)

In [ ]:
print(
    f"Median time from shutter close to last alert issued: {timing_stats['median_last_alert_time']:.1f} seconds"
)

print(
    f"Fraction of visits with all alerts issued in less than 120 seconds: {timing_stats['alert_frac_under_120s']:.3f}"
)

print(
    f"Fraction of visits with all alerts issued in less than 180 seconds: {timing_stats['alert_frac_under_180s']:.3f}"
)

Alert Generation Completeness
-----

In [ ]:
summed_stats = defaultdict(lambda: 0)

for day_obs in dayobs_range(f"{day_obs_start}", f"{day_obs_end}"):
    day_stats = await collect_visit_stats_for_dayobs(day_obs, client, b)
    for key in day_stats.keys():
        summed_stats[key] += day_stats[key]

In [ ]:
enabled_pp_detectors = 169
print(f"Number of observed science visits: {summed_stats['butler_exposure_count']}")

#
# This is NOT the number of detectors that generated alerts, since we don't
# have complete template coverage. PP still performs single frame processing
# on images without templates, so this counts detectors that successfully
# completed that.
#
success_frac = summed_stats["prompt_detector_count"] / (
    summed_stats["butler_exposure_count"] * enabled_pp_detectors
)
print(f"Fraction of detectors successfully processed in PP: {success_frac:.3f}")